# Предсказание пола по транзакциям — Sber 2026 DS

**Цель:** ROC AUC > 0.88 и Accuracy > 0.80

**Решение:** `solution_super_200.py` — super-ensemble: все типы фич (685 кандидатов) → top-200 → 5-fold LGB.

## 1. Установка зависимостей

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
        raise RuntimeError(f"Command failed: {cmd}")

run(f"{sys.executable} -m pip install -r requirements.txt -q")
print("Dependencies installed.")

## 2. Загрузка данных

Скачивает `train.csv`, `val.csv`, `test.csv` из HuggingFace в `./data/`.

In [ ]:
import os
os.makedirs("data", exist_ok=True)

if all(os.path.exists(f"data/{s}.csv") for s in ["train", "val", "test"]):
    print("Data already downloaded.")
else:
    run(f"{sys.executable} download_dataset.py")
    print("Data downloaded.")

for split in ["train", "val", "test"]:
    size = os.path.getsize(f"data/{split}.csv") // 1024 // 1024
    print(f"  data/{split}.csv  {size} MB")

## 3. Вычисление gender-embeddings (spaCy)

Считает косинусную близость описаний MCC/TR к мужским/женским словам.  
Сохраняет `results/gender_embeddings/mcc_gender.csv` и `tr_gender.csv`.

In [ ]:
if (os.path.exists("results/gender_embeddings/mcc_gender.csv") and
    os.path.exists("results/gender_embeddings/tr_gender.csv")):
    print("Gender embeddings already computed.")
else:
    # Download spaCy model if needed
    run(f"{sys.executable} -m spacy download ru_core_news_md -q")
    run(f"{sys.executable} compute_gender_embeddings.py")
    print("Gender embeddings computed.")

import pandas as pd
mcc_emb = pd.read_csv("results/gender_embeddings/mcc_gender.csv")
tr_emb  = pd.read_csv("results/gender_embeddings/tr_gender.csv")
print(f"MCC embeddings: {len(mcc_emb)} codes")
print(f"TR  embeddings: {len(tr_emb)} types")
display(mcc_emb.head())

## 4. Обучение модели

**Super-200:** все типы фич (685 кандидатов) → top-200 по LGB importance → 5-fold ensemble.

In [ ]:
import io
from contextlib import redirect_stdout

buf = io.StringIO()
with redirect_stdout(buf):
    exec(open("solution_super_200.py").read())
output = buf.getvalue()
print(output)

## 5. Итоговые метрики

In [ ]:
import json

with open("results/overall.json") as f:
    overall = json.load(f)

# Show super_200 result
entry = next((e for e in overall if e["branch"] == "solution_2026_04_15_super_200"), None)
if entry:
    m = entry["metrics"]
    t = entry.get("timing", {})
    auc = m["auc_score"]
    acc = m["accuracy_score"]
    print("=" * 40)
    print(f"  Model:     super_200  ({entry['n_features']} features)")
    print(f"  ROC AUC:   {auc}  {'✓' if auc > 0.88 else '✗'} (target > 0.88)")
    print(f"  Accuracy:  {acc}  {'✓' if acc > 0.80 else '✗'} (target > 0.80)")
    print(f"  Precision: {m['precision_score']}")
    print(f"  Recall:    {m['recall_score']}")
    if t:
        print(f"  Train:     {t.get('train_time_s')}s")
    print("=" * 40)

## 6. Сравнение всех экспериментов

In [ ]:
with open("results/overall.json") as f:
    overall = json.load(f)

rows = []
for e in overall:
    m = e["metrics"]
    rows.append({
        "branch": e["branch"].replace("solution_2026_04_15_", ""),
        "AUC": m["auc_score"],
        "Accuracy": m["accuracy_score"],
        "n_features": e.get("n_features", ""),
        "train_s": e.get("timing", {}).get("train_time_s", ""),
    })

df_res = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
df_res["AUC_ok"] = df_res["AUC"].apply(lambda x: "✓" if x > 0.88 else "")
df_res["Acc_ok"] = df_res["Accuracy"].apply(lambda x: "✓" if x > 0.80 else "")
display(df_res)